# 🧹 BƯỚC 2: Tiền xử lý dữ liệu (Preprocessing)
**Amazon Clothing Review Analysis & Recommendation**

---
### 🎯 Mục tiêu của Bước 2:
1. Load toàn bộ dữ liệu với **sampling thông minh** (5% ≈ ~1.1M reviews)
2. Làm sạch dữ liệu (xử lý missing, duplicate, outlier)
3. Chuẩn hóa kiểu dữ liệu
4. Tạo nhãn **Sentiment** từ rating
5. Merge review + meta
6. Lưu kết quả dạng `.parquet` để dùng cho các bước sau

### 💡 Tại sao dùng Sampling?
- File gốc: **22.6M reviews** (~5GB) → không thể load hết vào RAM
- Sample **5%** → ~**1.1M reviews** → đủ lớn để mô hình học tốt, đủ nhỏ để xử lý nhanh
- Sampling **ngẫu nhiên đều** → giữ được tính đại diện của dữ liệu

## Cell 2.1 — Import & Setup

In [ ]:
# ============================================================
# CELL 2.1: Import thư viện & cấu hình đường dẫn
# ============================================================

import pandas as pd
import numpy as np
import json, gzip, os, sys, re
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm         # Thanh tiến trình khi load data lớn

# Đường dẫn
ROOT_DIR      = Path().resolve().parent
REVIEW_PATH   = ROOT_DIR / "data" / "raw" / "Clothing_Shoes_and_Jewelry.jsonl.gz"
META_PATH     = ROOT_DIR / "data" / "raw" / "meta_Clothing_Shoes_and_Jewelry.jsonl.gz"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
SAMPLE_DIR    = ROOT_DIR / "data" / "sample"
FIGURES_DIR   = ROOT_DIR / "outputs" / "figures"

# Tạo thư mục nếu chưa có
for d in [PROCESSED_DIR, SAMPLE_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ============================================================
# SAMPLING CONFIG
# Thay đổi các tham số này nếu cần
# ============================================================
SAMPLE_RATE           = 0.05    # 5% review  (~1.1M dòng)
MAX_META_ROWS         = 500_000 # 500k sản phẩm
MIN_TEXT_LENGTH       = 10      # Review phải có ít nhất 10 ký tự
MIN_REVIEWS_PER_USER  = 5       # User phải có ít nhất 5 reviews (cho Recommendation)
MIN_REVIEWS_PER_ITEM  = 5       # Item phải có ít nhất 5 reviews
RANDOM_SEED           = 42

np.random.seed(RANDOM_SEED)
print("✅ Setup hoàn tất!")
print(f"   Sample rate  : {SAMPLE_RATE*100:.0f}% reviews")
print(f"   Max meta rows: {MAX_META_ROWS:,}")

## Cell 2.2 — Load Review với Sampling

In [ ]:
# ============================================================
# CELL 2.2: Load Review dataset với sampling ngẫu nhiên
#
# Cơ chế: mỗi dòng có SAMPLE_RATE% xác suất được chọn
# → kết quả là sample ngẫu nhiên đồng đều trên toàn dataset
# tqdm: hiển thị thanh tiến trình để biết đang load đến đâu
# ============================================================

def load_review_sampled(filepath, sample_rate=0.05):
    data = []
    total_read = 0

    print(f"📦 Loading review (sample {sample_rate*100:.0f}%)...")
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        for line in tqdm(f, desc="Reading", unit=" lines"):
            total_read += 1
            # Mỗi dòng có sample_rate xác suất được giữ lại
            if np.random.random() > sample_rate:
                continue
            try:
                data.append(json.loads(line.strip()))
            except:
                continue

    print(f"\n   Tổng dòng đã đọc : {total_read:,}")
    print(f"   Dòng được giữ lại: {len(data):,} ({len(data)/total_read*100:.2f}%)")
    return pd.DataFrame(data)

df_review_raw = load_review_sampled(REVIEW_PATH, SAMPLE_RATE)
print(f"\n✅ Review raw shape: {df_review_raw.shape}")

## Cell 2.3 — Load Meta dataset

In [ ]:
# ============================================================
# CELL 2.3: Load Meta dataset
# Meta nhỏ hơn nhiều so với review → load max_rows đầu
# ============================================================

def load_meta(filepath, max_rows=500_000):
    data = []
    print(f"📦 Loading meta (max {max_rows:,} rows)...")
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(tqdm(f, desc="Reading", unit=" lines")):
            if i >= max_rows:
                break
            try:
                data.append(json.loads(line.strip()))
            except:
                continue
    print(f"   Loaded: {len(data):,} rows")
    return pd.DataFrame(data)

df_meta_raw = load_meta(META_PATH, MAX_META_ROWS)
print(f"\n✅ Meta raw shape: {df_meta_raw.shape}")

## Cell 2.4 — Tiền xử lý Review

In [ ]:
# ============================================================
# CELL 2.4: Tiền xử lý Review - từng bước chi tiết
# ============================================================

def preprocess_review(df):
    df = df.copy()
    log = {}  # Ghi lại số dòng sau mỗi bước
    log['1_raw'] = len(df)

    # ----------------------------------------------------------
    # BƯỚC A: Chọn cột cần thiết
    # Chỉ giữ các cột có ý nghĩa cho phân tích
    # ----------------------------------------------------------
    needed = ['rating', 'text', 'title', 'user_id',
              'parent_asin', 'timestamp', 'helpful_vote', 'verified_purchase']
    existing_cols = [c for c in needed if c in df.columns]
    df = df[existing_cols]
    print(f"A. Chọn cột: giữ lại {existing_cols}")

    # ----------------------------------------------------------
    # BƯỚC B: Xử lý missing values
    # rating và text là BẮT BUỘC → drop nếu thiếu
    # các cột khác → fill giá trị mặc định
    # ----------------------------------------------------------
    df = df.dropna(subset=['rating', 'text'])
    df['helpful_vote']      = df.get('helpful_vote', pd.Series(0)).fillna(0)
    df['verified_purchase'] = df.get('verified_purchase', pd.Series(False)).fillna(False)
    log['2_drop_na'] = len(df)
    print(f"B. Drop NA  : {log['1_raw']:,} → {log['2_drop_na']:,} (-{log['1_raw']-log['2_drop_na']:,})")

    # ----------------------------------------------------------
    # BƯỚC C: Chuẩn hóa kiểu dữ liệu
    # ----------------------------------------------------------
    df['rating']        = pd.to_numeric(df['rating'], errors='coerce')
    df['helpful_vote']  = pd.to_numeric(df['helpful_vote'], errors='coerce').fillna(0).astype(int)
    df = df.dropna(subset=['rating'])  # drop nếu rating không parse được
    df['rating']        = df['rating'].astype(float)
    log['3_dtype'] = len(df)
    print(f"C. Dtype    : {log['2_drop_na']:,} → {log['3_dtype']:,}")

    # ----------------------------------------------------------
    # BƯỚC D: Chuyển timestamp → datetime
    # timestamp gốc là Unix milliseconds (số ms từ 1970)
    # ----------------------------------------------------------
    if 'timestamp' in df.columns:
        df['date']  = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
        df['year']  = df['date'].dt.year
        df['month'] = df['date'].dt.month
        print(f"D. Timestamp: range {df['year'].min():.0f} - {df['year'].max():.0f}")

    # ----------------------------------------------------------
    # BƯỚC E: Tạo nhãn Sentiment từ Rating
    # 4-5 sao → positive | 3 sao → neutral | 1-2 sao → negative
    # Đây là SUPERVISED LABEL cho bài toán Sentiment Analysis
    # ----------------------------------------------------------
    def to_sentiment(r):
        if r >= 4:   return 'positive'
        elif r == 3: return 'neutral'
        else:        return 'negative'

    df['sentiment']       = df['rating'].apply(to_sentiment)
    df['sentiment_score'] = df['rating'].apply(lambda r: 1 if r >= 4 else (0 if r == 3 else -1))
    print(f"E. Sentiment: {df['sentiment'].value_counts().to_dict()}")

    # ----------------------------------------------------------
    # BƯỚC F: Lọc review quá ngắn (noise)
    # Review < 10 ký tự không có giá trị phân tích NLP
    # ----------------------------------------------------------
    df['text_length'] = df['text'].astype(str).str.len()
    before = len(df)
    df = df[df['text_length'] >= MIN_TEXT_LENGTH]
    log['4_short_text'] = len(df)
    print(f"F. Short text: {before:,} → {log['4_short_text']:,} (-{before-log['4_short_text']:,} review quá ngắn)")

    # ----------------------------------------------------------
    # BƯỚC G: Loại bỏ duplicate
    # Cùng 1 user review cùng 1 sản phẩm → chỉ giữ lần đầu
    # ----------------------------------------------------------
    before = len(df)
    df = df.drop_duplicates(subset=['user_id', 'parent_asin'], keep='first')
    log['5_dedup'] = len(df)
    print(f"G. Duplicate: {before:,} → {log['5_dedup']:,} (-{before-log['5_dedup']:,} duplicates)")

    # ----------------------------------------------------------
    # BƯỚC H: Reset index
    # ----------------------------------------------------------
    df = df.reset_index(drop=True)

    print(f"\n✅ Kết quả cuối: {len(df):,} reviews")
    return df, log

df_review_clean, review_log = preprocess_review(df_review_raw)

## Cell 2.5 — Tiền xử lý Meta

In [ ]:
# ============================================================
# CELL 2.5: Tiền xử lý Meta dataset
# ============================================================

def preprocess_meta(df):
    df = df.copy()
    print(f"Raw shape: {df.shape}")

    # ----------------------------------------------------------
    # BƯỚC A: Chọn cột cần thiết
    # ----------------------------------------------------------
    needed = ['parent_asin', 'title', 'price', 'description',
              'categories', 'average_rating', 'rating_number',
              'store', 'main_category']
    existing_cols = [c for c in needed if c in df.columns]
    df = df[existing_cols]
    print(f"A. Giữ cột: {existing_cols}")

    # ----------------------------------------------------------
    # BƯỚC B: Drop sản phẩm không có ID hoặc tên
    # ----------------------------------------------------------
    before = len(df)
    df = df.dropna(subset=['parent_asin', 'title'])
    df = df.drop_duplicates(subset=['parent_asin'], keep='first')
    print(f"B. Drop NA/dup: {before:,} → {len(df):,}")

    # ----------------------------------------------------------
    # BƯỚC C: Xử lý Price
    # Giá có thể là '$29.99' hoặc '29.99' → cần chuẩn hóa
    # Lọc outlier: giá <= 0 hoặc > $10,000 là bất thường
    # ----------------------------------------------------------
    if 'price' in df.columns:
        df['price'] = df['price'].astype(str).str.replace(r'[^\d.]', '', regex=True)
        df['price'] = pd.to_numeric(df['price'], errors='coerce')
        valid_price = df['price'].between(0.01, 10_000)
        print(f"C. Price: {valid_price.sum():,} sản phẩm có giá hợp lệ / {len(df):,}")
        # Giữ cả null price (không xóa sản phẩm, chỉ để null)

    # ----------------------------------------------------------
    # BƯỚC D: Xử lý Description (list → string)
    # description gốc là list các đoạn văn
    # ----------------------------------------------------------
    if 'description' in df.columns:
        df['description'] = df['description'].apply(
            lambda x: ' '.join(x) if isinstance(x, list) else str(x) if pd.notna(x) else ''
        )
        print(f"D. Description: đã chuyển list → string")

    # ----------------------------------------------------------
    # BƯỚC E: Xử lý Categories
    # categories là list lồng nhau → lấy level 1 làm main_category
    # ----------------------------------------------------------
    if 'categories' in df.columns:
        def extract_category(cats):
            if isinstance(cats, list) and len(cats) > 0:
                inner = cats[0] if isinstance(cats[0], list) else cats
                return inner[0] if len(inner) > 0 else 'Unknown'
            return 'Unknown'
        df['main_category'] = df['categories'].apply(extract_category)
        print(f"E. Categories top 5: {df['main_category'].value_counts().head().to_dict()}")

    df = df.reset_index(drop=True)
    print(f"\n✅ Meta kết quả: {df.shape}")
    return df

df_meta_clean = preprocess_meta(df_meta_raw)

## Cell 2.6 — Lọc theo số lượng review (Cold Start Filter)

In [ ]:
# ============================================================
# CELL 2.6: Cold-start filter cho Recommendation System
#
# Vấn đề Cold Start: user/item có QUÁ ÍT reviews
# → model không học được preference của họ
# Giải pháp: lọc chỉ giữ user và item có đủ interactions
# ============================================================

print(f"Trước khi filter:")
print(f"  Reviews : {len(df_review_clean):,}")
print(f"  Users   : {df_review_clean['user_id'].nunique():,}")
print(f"  Items   : {df_review_clean['parent_asin'].nunique():,}")

df_filtered = df_review_clean.copy()

# Lọc nhiều vòng lặp cho đến khi ổn định
# (vì sau khi lọc user, item có thể mất bớt; sau khi lọc item, user có thể mất bớt)
for iteration in range(5):
    n_before = len(df_filtered)

    # Lọc item: chỉ giữ item có >= MIN_REVIEWS_PER_ITEM reviews
    item_counts = df_filtered['parent_asin'].value_counts()
    valid_items = item_counts[item_counts >= MIN_REVIEWS_PER_ITEM].index
    df_filtered = df_filtered[df_filtered['parent_asin'].isin(valid_items)]

    # Lọc user: chỉ giữ user có >= MIN_REVIEWS_PER_USER reviews
    user_counts = df_filtered['user_id'].value_counts()
    valid_users = user_counts[user_counts >= MIN_REVIEWS_PER_USER].index
    df_filtered = df_filtered[df_filtered['user_id'].isin(valid_users)]

    n_after = len(df_filtered)
    print(f"  Iter {iteration+1}: {n_before:,} → {n_after:,}")
    if n_before == n_after:  # Ổn định rồi, dừng
        break

print(f"\nSau khi filter:")
print(f"  Reviews : {len(df_filtered):,}")
print(f"  Users   : {df_filtered['user_id'].nunique():,}")
print(f"  Items   : {df_filtered['parent_asin'].nunique():,}")

# Lưu cả 2 version:
# - df_review_clean: dùng cho Sentiment Analysis (không cần filter cold start)
# - df_filtered    : dùng cho Recommendation System
df_review_for_rec = df_filtered.copy()
print("\n✅ Đã tạo:")
print(f"   df_review_clean   → Sentiment ({len(df_review_clean):,} rows)")
print(f"   df_review_for_rec → Recommendation ({len(df_review_for_rec):,} rows)")

## Cell 2.7 — Merge Review + Meta

In [ ]:
# ============================================================
# CELL 2.7: Merge Review và Meta theo parent_asin
#
# Dùng LEFT JOIN: giữ tất cả reviews, thêm thông tin sản phẩm
# Nếu sản phẩm không có trong meta → các cột meta sẽ là NaN
# ============================================================

# Chọn cột meta cần merge (không cần merge tất cả)
meta_cols_to_merge = ['parent_asin', 'title', 'price', 'main_category']
meta_cols_available = [c for c in meta_cols_to_merge if c in df_meta_clean.columns]

df_merged = df_review_clean.merge(
    df_meta_clean[meta_cols_available],
    on='parent_asin',
    how='left'
)

# Đổi tên cột title từ meta để không nhầm với title review
if 'title_x' in df_merged.columns:
    df_merged = df_merged.rename(columns={'title_x': 'review_title', 'title_y': 'product_title'})
elif 'title' in df_merged.columns and 'title' in df_review_clean.columns:
    pass  # không bị conflict

print(f"✅ Merged shape: {df_merged.shape}")
print(f"   Match rate  : {df_merged['main_category'].notna().mean()*100:.1f}% reviews có thông tin meta")
display(df_merged.head(3))

## Cell 2.8 — Lưu kết quả

In [ ]:
# ============================================================
# CELL 2.8: Lưu các file đã xử lý
#
# Dùng Parquet thay vì CSV vì:
#   - Nhanh hơn đọc/ghi ~5-10x
#   - Nén tốt hơn (~3-5x nhỏ hơn CSV)
#   - Giữ nguyên kiểu dữ liệu (datetime, int...)
# ============================================================

files_to_save = {
    PROCESSED_DIR / "review_clean.parquet"    : df_review_clean,
    PROCESSED_DIR / "meta_clean.parquet"      : df_meta_clean,
    PROCESSED_DIR / "merged_clean.parquet"    : df_merged,
    PROCESSED_DIR / "review_for_rec.parquet"  : df_review_for_rec,
}

print("💾 Đang lưu file...")
for path, df in files_to_save.items():
    df.to_parquet(path, index=False)
    size_mb = path.stat().st_size / (1024**2)
    print(f"  ✅ {path.name:35s} | {len(df):>10,} rows | {size_mb:.1f} MB")

# Tạo thêm file sample nhỏ (10k rows) để test nhanh ở các bước sau
sample_review = df_review_clean.sample(n=min(10_000, len(df_review_clean)), random_state=42)
sample_review.to_parquet(SAMPLE_DIR / "review_sample.parquet", index=False)
print(f"  ✅ {'review_sample.parquet':35s} | {len(sample_review):>10,} rows (sample)")

print("\n✅ Đã lưu tất cả file vào data/processed/ và data/sample/")

## Cell 2.9 — Kiểm tra chất lượng & Báo cáo tổng kết

In [ ]:
# ============================================================
# CELL 2.9: Visualize kết quả preprocessing
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Bước 2 - Kết quả sau Preprocessing', fontsize=14, fontweight='bold')

# ---- Biểu đồ 1: Phân bố Sentiment ----
sentiment_counts = df_review_clean['sentiment'].value_counts()
colors_sent = {'positive': '#2ecc71', 'neutral': '#f39c12', 'negative': '#e74c3c'}
bars = axes[0,0].bar(sentiment_counts.index,
                      sentiment_counts.values,
                      color=[colors_sent.get(s, 'gray') for s in sentiment_counts.index])
axes[0,0].set_title('Phân bố Sentiment Label')
axes[0,0].set_ylabel('Số lượng')
for bar, val in zip(bars, sentiment_counts.values):
    pct = val / len(df_review_clean) * 100
    axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                    f'{pct:.1f}%', ha='center', fontsize=10, fontweight='bold')

# ---- Biểu đồ 2: Phân bố độ dài review (sau khi lọc) ----
text_len = df_review_clean['text_length'].clip(upper=1000)
axes[0,1].hist(text_len, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0,1].set_title('Phân bố độ dài Review (sau lọc)')
axes[0,1].set_xlabel('Số ký tự (cắt tại 1000)')
axes[0,1].set_ylabel('Tần suất')
axes[0,1].axvline(df_review_clean['text_length'].median(), color='red',
                    linestyle='--', label=f'Median: {df_review_clean["text_length"].median():.0f}')
axes[0,1].legend()

# ---- Biểu đồ 3: Review theo năm ----
if 'year' in df_review_clean.columns:
    year_counts = df_review_clean['year'].value_counts().sort_index()
    year_counts = year_counts[year_counts.index.between(2000, 2024)]
    axes[1,0].bar(year_counts.index.astype(int), year_counts.values, color='#3498db', alpha=0.8)
    axes[1,0].set_title('Số Review theo Năm')
    axes[1,0].set_xlabel('Năm')
    axes[1,0].set_ylabel('Số lượng')
    axes[1,0].tick_params(axis='x', rotation=45)

# ---- Biểu đồ 4: Data pipeline flow ----
axes[1,1].axis('off')
steps = [
    f"Raw Review\n{len(df_review_raw):,} rows",
    f"Drop NA\n{review_log.get('2_drop_na', 0):,} rows",
    f"Filter short text\n{review_log.get('4_short_text', 0):,} rows",
    f"Remove duplicate\n{review_log.get('5_dedup', 0):,} rows",
    f"Final Clean\n{len(df_review_clean):,} rows",
]
y_positions = [0.9, 0.72, 0.54, 0.36, 0.18]
colors_flow = ['#95a5a6', '#3498db', '#3498db', '#3498db', '#2ecc71']
for step, y, color in zip(steps, y_positions, colors_flow):
    axes[1,1].text(0.5, y, step, ha='center', va='center', fontsize=9,
                    bbox=dict(boxstyle='round,pad=0.4', facecolor=color, alpha=0.7),
                    transform=axes[1,1].transAxes)
    if y > 0.18:
        axes[1,1].annotate('', xy=(0.5, y - 0.11), xytext=(0.5, y - 0.05),
                             xycoords='axes fraction', textcoords='axes fraction',
                             arrowprops=dict(arrowstyle='->', color='black'))
axes[1,1].set_title('Pipeline Flow - Review')

plt.tight_layout()
save_path = FIGURES_DIR / 'step2_preprocessing_result.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 Đã lưu: {save_path}")

In [ ]:
# ============================================================
# CELL 2.10: Tổng kết Bước 1 & 2
# ============================================================

print("=" * 60)
print("🎉 TỔNG KẾT BƯỚC 1 & 2")
print("=" * 60)

print("\n📦 FILE ĐÃ LƯU:")
for f in PROCESSED_DIR.glob("*.parquet"):
    size = f.stat().st_size / (1024**2)
    print(f"  data/processed/{f.name:<35} ({size:.1f} MB)")
for f in SAMPLE_DIR.glob("*.parquet"):
    size = f.stat().st_size / (1024**2)
    print(f"  data/sample/{f.name:<35} ({size:.1f} MB)")

print("\n📊 THỐNG KÊ SAU XỬ LÝ:")
print(f"  df_review_clean   : {len(df_review_clean):>10,} reviews (dùng cho Sentiment)")
print(f"  df_review_for_rec : {len(df_review_for_rec):>10,} reviews (dùng cho Recommendation)")
print(f"  df_meta_clean     : {len(df_meta_clean):>10,} sản phẩm")
print(f"  df_merged         : {len(df_merged):>10,} (review + meta)")
print(f"\n  Users (rec)       : {df_review_for_rec['user_id'].nunique():>10,}")
print(f"  Items (rec)       : {df_review_for_rec['parent_asin'].nunique():>10,}")
print(f"\n  Sentiment phân bố :")
for s, c in df_review_clean['sentiment'].value_counts().items():
    pct = c/len(df_review_clean)*100
    print(f"    {s:10}: {c:>8,} ({pct:.1f}%)")

print("\n🚀 Sẵn sàng chuyển sang BƯỚC 3: EDA chuyên sâu!")
print("   → Load file: pd.read_parquet('data/processed/merged_clean.parquet')")